# SASHIMI-C: a Milky Way subhalo population

## Goal
Compute a CDM population in a $10^{12}M_\odot$ host at $z=0$: the subhalo mass
function before and after stripping, satellite counts under two explicit formation
cuts, a resolved annihilation contribution, and one Poisson realization. These
are the main observables illustrated by the historical `sample.ipynb`.

This is a physical example of the current ITAMAE-backed calculation. Each figure
is computed below from the selected model, with explicit units and population
cuts. A joint grid refinement at the end measures numerical sensitivity for this
example; it does not establish simulation calibration or an observational limit.

## Setup
From this repository's migration checkout, install the pinned development environment:
```sh
uv sync --extra demo
uv run --no-sync python -m ipykernel install --user --name sashimi-c-demo --display-name "sashimi-c demo"
```
Select that kernel and **Restart Kernel and Run All**. Allow several minutes for
the population and refinement calculations. No input files are downloaded by the
cells. The output directory is local to the notebook. Historical examples are
preserved in [archive/](archive/); the independent migration audit is in
[scientific_validation.ipynb](scientific_validation.ipynb).

In [ ]:
%matplotlib inline
import sys
import json
import time
import warnings
from pathlib import Path
from collections import Counter
from importlib.metadata import version
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from itamae.provenance import source_revision
from itamae.types import WeightedSubhaloCatalog
import itamae

plt.rcParams.update({"figure.figsize": (7.2, 4.5), "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.18,
                     "figure.constrained_layout.use": True})
COLORS = ["#0072B2", "#D55E00", "#009E73", "#555555"]
STYLES = ["-", "--", "-.", ":"]
output_dir = Path("outputs/usage_walkthrough")
output_dir.mkdir(parents=True, exist_ok=True)


def table(headers, rows):
    display(Markdown("| " + " | ".join(headers) + " |\n| " +
                     " | ".join(["---"] * len(headers)) + " |\n" +
                     "\n".join("| " + " | ".join(map(str, row)) + " |" for row in rows)))


def calculate(label, function):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as observed:
        warnings.simplefilter("always", RuntimeWarning)
        result = function()
    print(f"{label}: {time.perf_counter() - start:.1f} s")
    if observed:
        counts = Counter(f"{item.category.__name__}: {item.message}" for item in observed)
        table(["Recorded warning", "Occurrences"], counts.items())
    return result


def check_catalog(catalog):
    weights = catalog.weight_final
    assert np.all(np.isfinite(weights)) and np.all(weights >= 0)
    assert weights.sum() > 0
    mass = catalog.columns["m_bound"]
    assert np.all(np.isfinite(mass)) and np.all(mass >= 0)
    return {"nodes": len(catalog), "expected_survivors": float(weights.sum()),
            "bound_mass_fraction": float(catalog.weighted_sum(mass) / 1e12),
            "N_bound_gt_1e8": float(weights[mass > 1e8].sum())}


def mass_function(catalog, edges, selection=None, column="m_bound"):
    selected = catalog if selection is None else catalog.select(selection)
    counts, _ = selected.weighted_histogram(column, bins=edges)
    return np.sqrt(edges[:-1] * edges[1:]), counts / np.diff(np.log(edges))



def accretion_bin_edges(catalog, group=6):
    # Bin boundaries lie between accretion-grid nodes, avoiding bin/grid beating.
    log_nodes = np.log(np.unique(catalog.columns["m200_acc"]))
    spacing = np.diff(log_nodes)
    np.testing.assert_allclose(spacing, spacing[0], rtol=1e-10)
    interior = (log_nodes[:-1] + log_nodes[1:]) / 2
    return np.exp(np.r_[log_nodes[0] - spacing[0]/2, interior[group-1::group],
                        log_nodes[-1] + spacing[-1]/2])


def cumulative(values, weights, thresholds):
    return np.array([weights[values > threshold].sum() for threshold in thresholds])


def save_catalog(catalog, name):
    path = output_dir / (name + ".npz")
    catalog.to_npz(path)
    restored = WeightedSubhaloCatalog.from_npz(path)
    np.testing.assert_array_equal(restored.weight_final, catalog.weight_final)
    for column in catalog.columns:
        np.testing.assert_array_equal(restored.columns[column], catalog.columns[column])
    assert restored.metadata == catalog.metadata
    return path

import sashimi_c
components = [("sashimi-itamae", "itamae", itamae), ("sashimi-c", "sashimi-c", sashimi_c)]
provenance = {dist: {"version": version(dist), "source_revision": source_revision(name, module_file=mod.__file__)}
              for dist, name, mod in components}
table(["Component", "Version", "Source revision"],
      [(name, info["version"], info["source_revision"]) for name, info in provenance.items()])
print("Python", sys.version.split()[0], "| NumPy", version("numpy"), "| SciPy", version("scipy"))

## 1. Choose the host and calculate its population

The accretion grid covers $10^5$–$10^{11}M_\odot$ in $M_{200}$ and $0<z_{acc}\le7$.
This finite population includes dwarf-scale progenitors. The default no-disruption
threshold `ct_th=0` and `pert2_shanks` stripping prescription are explicit.
The histogram below $10^6M_\odot$ is omitted because stripping can move objects
below the input mass floor. No completeness correction is applied there.

In [ ]:
from sashimi_c import SubhaloObservables
parameters = dict(M0_per_Msun=1e12, redshift=0., dz=.05, zmax=7.,
                  N_ma=192, N_herm=5, N_hermNa=64, sigmalogc=.128,
                  logmamin=5., logmamax=11., ct_th=0., Na_model=3,
                  profile_change=True, method="pert2_shanks")
observables = calculate("CDM population", lambda: SubhaloObservables(**parameters))
catalog = observables.catalog
metrics = check_catalog(catalog)
table(["Quantity", "Value"], [(name, f"{value:.6g}") for name, value in metrics.items()])
print("Physics:", catalog.metadata["calculation_specification"])
print("Units:", catalog.metadata["canonical_units"])
mass_edges = np.geomspace(1e6, 1e11, 31)

## 2. See the effect of tidal stripping

Both curves describe the **same surviving population**. The accretion curve uses
its original $M_{200,acc}$ coordinate; the bound-mass curve uses the mass inside
the current tidal radius. It is not a comparison with all disrupted progenitors.
The display bins group adjacent accretion nodes to avoid bin/grid beating.
The ordinate $m\,dN/d\ln m=m^2dN/dm$ is mass per natural-log mass interval.

In [ ]:
fig, ax = plt.subplots()
for evolved, label, style in [(False, "Accretion mass of survivors", "--"),
                               (True, "Bound mass at z = 0", "-")]:
    mass, dndlnm = mass_function(catalog, accretion_bin_edges(catalog),
                                column="m_bound" if evolved else "m200_acc")
    plotted = (mass >= mass_edges[0]) & (mass <= mass_edges[-1]) & (dndlnm > 0)
    ax.loglog(mass[plotted], (mass*dndlnm)[plotted], style, label=label)
ax.set(xlabel=r"Subhalo mass [$M_\odot$]", ylabel=r"$m\,dN/d\ln m$ [$M_\odot$]",
       title=r"CDM in a $10^{12}M_\odot$ host")
ax.legend()
plt.show()

## The $V_{max}$–$r_{max}$ distribution

Select surviving subhalos with current bound mass $>10^8M_\odot$ and sum their
expected-count weights into bins. The axes use km/s and kpc; the color is count
per dex². $r_{max}=2.163r_s$ and $V_{max}$ are the model's NFW structural parameters,
which are retained even when the tidal radius lies inside $r_{max}$.

In [ ]:
selected = observables.m0 / observables.Msun > 1e8
structure_views = [("CDM", observables.Vmax[selected]/(observables.km/observables.s),
                    observables.rmax[selected]/observables.kpc, observables.weight[selected])]

from matplotlib.colors import LogNorm
v_edges = np.geomspace(3, 180, 32)
r_edges = np.geomspace(.02, 80, 32)
structure_histograms = []
for label, vx, ry, weights in structure_views:
    counts, _, _ = np.histogram2d(vx, ry, bins=(v_edges, r_edges), weights=weights)
    area = np.diff(np.log10(v_edges))[:, None] * np.diff(np.log10(r_edges))[None, :]
    structure_histograms.append(counts / area)
    print(f"{label}: {weights.sum():.3f} selected; {counts.sum():.3f} inside displayed axes")
maximum = max(hist.max() for hist in structure_histograms)
fig, axes = plt.subplots(1, len(structure_views), figsize=(6*len(structure_views), 4.6), squeeze=False)
for ax, (label, _, _, _), hist in zip(axes[0], structure_views, structure_histograms):
    mesh = ax.pcolormesh(v_edges, r_edges, np.ma.masked_less_equal(hist.T, 0),
                         norm=LogNorm(vmin=max(maximum*1e-4, 1e-4), vmax=maximum), cmap="viridis")
    ax.set(xscale="log", yscale="log", xlabel=r"$V_{max}$ [km/s]", ylabel=r"$r_{max}$ [kpc]", title=label)
fig.colorbar(mesh, ax=axes[0].tolist(), label="Expected subhalos per dex²", shrink=.9)
fig.suptitle(r"Surviving structure at $m_{bound}>10^8 M_\odot$")
plt.show()

## 3. Apply two satellite-formation assumptions

The cuts are $M_{peak}>10^8M_\odot$ and $V_{peak}>18$ km/s. They are illustrative
occupation prescriptions, not observed galaxy counts or survey completeness.
The public observable methods accept thresholds in the model's internal units;
returned masses and velocities are in solar masses and km/s. Their histogram
curves are approximate cumulative displays; the table gives direct weight sums.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
cuts = [("Peak mass > 1e8 Msun", observables.ma200 > 1e8*observables.Msun,
         observables.Nsat_Mpeak(1e8*observables.Msun)),
        ("Peak velocity > 18 km/s", observables.Vpeak > 18*observables.km/observables.s,
         observables.Nsat_Vpeak(18*observables.km/observables.s))]
rows = []
for (label, selection, values), color, style in zip(cuts, COLORS, STYLES):
    mass, nm, velocity, nv = values
    rows.append((label, f"{observables.weight[selection].sum():.3f}"))
    for ax, x, y in [(axes[0], mass, nm), (axes[1], velocity, nv)]:
        ax.plot(x[y>0], y[y>0], style, color=color, label=label)
axes[0].set(xscale="log", yscale="log", xlabel=r"Bound mass [$M_\odot$]", ylabel=r"$N(>m)$")
axes[1].set(xscale="log", yscale="log", xlabel=r"$V_{max}$ [km/s]", ylabel=r"$N(>V_{max})$")
axes[0].legend(fontsize=9)
fig.suptitle("Expected satellites for the selected occupation rule")
plt.show()
table(["Formation cut", "Expected satellites"], rows)

## 4. Compute the resolved annihilation contribution

`n=0` includes first-level subhalos. Because the input floor is
$M_{200,acc}=10^5M_\odot$, this is a **resolved contribution**, not an extrapolation
to microhalos or a complete annihilation prediction. The implementation returns
$L_{total}/L_{host,0}=1-f_{sh}^2+B_{sh}$; that existing convention is retained.
Prompt-cusp and higher-order boost tables are separate inputs and are not used here.

In [ ]:
boost, luminosity_ratio = observables.annihilation_boost_factor(n=0)
fraction = observables.mass_fraction()
assert boost >= 0 and np.isfinite(boost)
np.testing.assert_allclose(fraction, metrics["bound_mass_fraction"], rtol=1e-13)
np.testing.assert_allclose(luminosity_ratio, 1-fraction**2+boost, rtol=1e-13)
table(["Resolved quantity", "Value"], [("Bound mass fraction", f"{fraction:.6g}"),
                                      ("B_sh, first-level only", f"{boost:.6g}"),
                                      ("L_total / L_host,0", f"{luminosity_ratio:.6g}")])

## 5. Draw one realization and inspect its density structure

The catalog rows are integration nodes with expected-count weights. A Poisson
realization turns that intensity into individual subhalos. The fixed seed makes
this example repeatable; scatter between realizations is not a measurement error.
The realization below selects current bound masses above $10^8M_\odot$.

In [ ]:
realization = observables.subhalo_catalog_MC(1e8*observables.Msun, seed=20260911)
ma, zacc, rsa, rhosa, mbound, rs, rhos, ct = realization
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
thresholds = np.geomspace(1e8, 1e11, 70)
expected = cumulative(catalog.columns["m_bound"], catalog.weight_final, thresholds)
drawn = cumulative(mbound, np.ones(len(mbound)), thresholds)
axes[0].step(thresholds, expected, where="post", label="Expected count")
axes[0].step(thresholds, drawn, where="post", ls="--", label="One Poisson draw")
axes[0].set(xscale="log", xlabel=r"Bound mass [$M_\odot$]", ylabel=r"$N(>m)$")
axes[0].legend()
points = axes[1].scatter(rs, rhos, c=np.log10(mbound), s=18, cmap="viridis")
axes[1].set(xscale="log", yscale="log", xlabel=r"$r_s$ [kpc]", ylabel=r"$\rho_s$ [$M_\odot$ pc$^{-3}$]")
axes[1].set_xticks([.5, 1, 2, 5], labels=["0.5", "1", "2", "5"])
axes[1].xaxis.set_minor_formatter(plt.NullFormatter())
fig.colorbar(points, ax=axes[1], label=r"$\log_{10}(m_{bound}/M_\odot)$")
fig.suptitle("Resolved CDM structure: one realization")
plt.show()
print(f"Drawn objects: {len(mbound)}; expected above the same bound-mass cut: {expected[0]:.3f}")

## Checks: refine the numerical grid at this host

We halve `dz`, increase the mass grid and both Hermite orders together, and keep
all physical inputs unchanged. The output table reports the remaining change.
Threshold counts can respond to nodes crossing the cut. This two-grid comparison
is local numerical evidence; for a precision application, continue the refinement
and compare the stripping solver as described in [resolution and states](../docs/resolution-and-states.md).

In [ ]:
refined_parameters = {**parameters, "dz": .025, "N_ma": 256, "N_herm": 7, "N_hermNa": 96}
refined = calculate("Refined CDM population", lambda: SubhaloObservables(**refined_parameters))
refined_catalog = refined.catalog

coarse_metrics, fine_metrics = check_catalog(catalog), check_catalog(refined_catalog)
refinement_rows = []
for metric in ("expected_survivors", "bound_mass_fraction", "N_bound_gt_1e8"):
    first, second = coarse_metrics[metric], fine_metrics[metric]
    delta = 100 * (second / first - 1)
    refinement_rows.append((metric, f"{first:.6g}", f"{second:.6g}", f"{delta:+.3f}%"))
table(["Quantity", "Displayed grid", "Refined grid", "Change"], refinement_rows)
fig, ax = plt.subplots()
for cat, label, style in [(catalog, "Displayed grid", "-"), (refined_catalog, "Refined grid", "--")]:
    mass, dndlnm = mass_function(cat, mass_edges)
    ax.loglog(mass, np.where(dndlnm > 0, dndlnm, np.nan), style, label=label)
ax.set(xlabel=r"Bound mass [$M_\odot$]", ylabel=r"$dN/d\ln m$", title="Sensitivity to joint numerical refinement")
ax.legend()
plt.show()
report = {"provenance": provenance, "parameters": parameters, "refined_parameters": refined_parameters,
          "displayed": coarse_metrics, "refined": fine_metrics}
(output_dir / "resolution-summary.json").write_text(json.dumps(report, indent=2) + "\n")
print("Catalog saved and checked:", save_catalog(catalog, "catalog"))
print("Refined catalog saved and checked:", save_catalog(refined_catalog, "catalog-refined"))

## Next steps

The computed curves, counts, boost contribution and sampled structures are the
ITAMAE-backed replacement for the corresponding legacy example. The saved
catalog retains units, parameters and source identities. Extend the accretion
mass/redshift domain and establish an observable-specific error budget before
using these values as precision predictions. Baryons and an observational
selection function are not included.

Model context: [Hiroshima, Ando & Ishiyama (2018)](https://arxiv.org/abs/1803.07691).
The [scientific validation notebook](scientific_validation.ipynb) records the
independent migration comparison and the retained solver limitations.